# import library

In [1]:
from appgeopy import *
from my_packages import *

Cannot find header.dxf (GDAL_DATA is not defined)


# Export information of geodetic data sets

## GPS data

### move GPS stations within InSAR frame into our working directory

### export start date, end date, coordinates

## MLCW data

## InSAR data

# MLCW - GPS pairs

## Find closest neighbors
First we need to find the closet GPS station to each MLCW station

## Move GPS files to current directory

## Only keep the GPS timeseries last longer than 5 years

## Model GPS dU timeseries

### model GPS dU using `gps_decompose.py`

**(Deprecated)**
This is my previous program, when I haven't enhanced it with `w-stat` 

### model GPS dU using `appsigsolv`

I have run `python batch_process_GPS_dU.py`, which activate the `python -m appsigsolv decompose` to model the raw GPS timeseries. then I will extract the primary signals and save the file in `GPS_modeled` folder

## Model MLCW

### run `appsigsolv`

**2026/05/02**:

I want to export these timeseries into CSV file so that `appsigsolv` can model the signal at each ring (run again on 2026/5/8)

### Export modeled signal to a new folder

## Check timeseries modeled GPS and MLCW

2026/05/03:

Now I need to compare the timeseries between modeled GPS and modeled MLCW (sum all rings),
then get the compaction percentage MLCW over GPS

**check MLCW total compaction over GPS**

**check MLCW ring-by-ring over GPS**

## Extract MLCW trend

calculate the trend of modeled MLCW data, but only select station with data from 2010 to 2021, more than 5 years.

## Get GPS velocity (Kriging) into MLCW points

**v1**

**v2**

# Extract GPS trend (2025/5/7)

Then perform interpolation with CoKriging, using leveling-derived slope as secondary data set

# Turn IDW interpolated shapefiles to feather files

# Extract 5-m regular time series from 2015 onward

# Interpolate MLCW/GPS ratio to short-time points

the file `MLCW_GPS_velocity_TWD97_v2.csv` only contains 33 points, because I exclude the stations with observation time < 5 years

therefore, I interpolated the values of ratio from `MLCW_GPS_velocity_TWD97_v2.shp` using radial basis function, method file saved in `ratio_RBF.lyr`

# MLCW and InSAR tasks

## Add coordinates into alpha ratio output from `alpha_insar_test.py`

## Turn to shapefile

# GROUNDWATER LEVEL DATA SET

# group MLCW by aquifers/aquitards

## group the reconstructed timeseries

to reduce the complexity of the problem, we will try to group magnetic rings into group

In [2]:
# ==============================================================================
# 1. LOAD MASTER MLCW STATION METADATA FROM HDF5
# ==============================================================================

# File path to the pre-processed, imputed Multilayer Compaction Well (MLCW) master dataset
h5_fpath = r"D:\1000_SCRIPTS\003_Project002\20251111_GTWR003\1_PrepareDatasets\MLCW_3\20251230_MLCW_CRFP_Imputed_v4.h5"

# Load the HDF5 data payload and metadata attributes using specialized groundwater tools
mlcw_data, mlcw_info = gwatertools.open_HDF5(h5_fpath)

# Extract valid station identifiers (keys containing dictionary structures) from the HDF5 file
available_stations = [
    ele for ele in mlcw_data.keys() if isinstance(mlcw_data[ele], dict)
]
# Preview the first 5 available groundwater monitoring stations (e.g., TUKU)
available_stations[:5]

['ANHE', 'ANNAN', 'BEICHEN', 'CANLIN', 'DONGGUANG']

In [3]:
# ==============================================================================
# 2. LOCATE RECONSTRUCTED TIMESERIES DATA (InSAR-MLCW COUPLING PROJECT)
# ==============================================================================

# Folder containing reconstructed MLCW time-series data files associated with InSAR processing
# mlcw_reconst_fld = r"D:\1000_SCRIPTS\004_Project003\20260427_InSAR_MLCW_v2\data\mlcw\reconstructed"

# 2026-5-27, change to modeled timeseries, to keep the original timeline
# mlcw_reconst_fld = (
#     r"D:\1000_SCRIPTS\004_Project003\20260427_InSAR_MLCW_v2\data\mlcw\modeled"
# )

# 2026-6-1
mlcw_reconst_fld = r"D:\1000_SCRIPTS\004_Project003\20260427_InSAR_MLCW_v2\data\mlcw\modeled_nojump\nojump"

# ------------------------------------------------------------------------------
# PURPOSE: DEFINE OUTPUT DIRECTORY AND ENSURE STORAGE PATH EXISTS
# ------------------------------------------------------------------------------

# Define the destination folder for storing aggregated layer-by-layer compaction datasets
# output_savefld = r"D:\1000_SCRIPTS\004_Project003\20260427_InSAR_MLCW_v2\data\mlcw\group_byLayer_modeled"
output_savefld = r"D:\1000_SCRIPTS\004_Project003\20260427_InSAR_MLCW_v2\data\mlcw\modeled_nojump\group_byLayer_nojump"

# Create the target directory automatically if it does not already exist
if not os.path.exists(output_savefld):
    os.makedirs(output_savefld, exist_ok=True)

# Gather paths of all reconstructed station text files (.csv format) inside the target directory
mlcw_reconst_files = glob(os.path.join(mlcw_reconst_fld, "*csv"))
# Quick check on the first 3 file paths
mlcw_reconst_files[:3]

['D:\\1000_SCRIPTS\\004_Project003\\20260427_InSAR_MLCW_v2\\data\\mlcw\\modeled_nojump\\nojump\\ANHE_ringbyring.csv',
 'D:\\1000_SCRIPTS\\004_Project003\\20260427_InSAR_MLCW_v2\\data\\mlcw\\modeled_nojump\\nojump\\ANNAN_ringbyring.csv',
 'D:\\1000_SCRIPTS\\004_Project003\\20260427_InSAR_MLCW_v2\\data\\mlcw\\modeled_nojump\\nojump\\BEICHEN_ringbyring.csv']

In [4]:
# ------------------------------------------------------------------------------
# PURPOSE: BATCH-PROCESS ALL AVAILABLE STATIONS ACROSS THE CHOUSHUI RIVER ALLUVIAL FAN
# ------------------------------------------------------------------------------

# Iterate systematically through every available monitoring station using a progress bar
for select_station in tqdm(available_stations):
    try:
        # Match and retrieve the reconstructed time-series CSV file path for the active station
        mlcw_reconst_fpath = [
            f for f in mlcw_reconst_files if select_station in f
        ][0]

        # Extract depth values and binary-encoded layer information from the station's HDF5 schema
        classify_byStation = mlcw_data[select_station]["classify"]
        depth_arr = np.round(classify_byStation["depth"], 3)

        # Decode binary/byte layer names into standard, readable UTF-8 strings
        layer_arr = [ele.decode("utf-8") for ele in classify_byStation["layer"]]

        # Construct a reference dataframe mapping sensor anchor depths to specific hydrogeological layers
        classify_df = pd.DataFrame(
            data={"depth": depth_arr, "layer": layer_arr}
        )
        classify_df = classify_df.astype("str")

        # Generate an ordered sequence of unique aquifer/aquitard layers monitored at this well site
        all_layers = classify_df["layer"].unique()

        # ------------------------------------------------------------------------------
        # PURPOSE: AGGREGATE SENSOR COMPACTION DATA BY HYDROGEOLOGICAL LAYERS
        # ------------------------------------------------------------------------------

        # Load the station's historical timeseries data, initializing the timestamp column as a DatetimeIndex
        mlcw_reconst_df = pd.read_csv(
            mlcw_reconst_fpath, index_col=[0], parse_dates=[0]
        )

        # Instantiate a tracking list for processed depths and a fresh container for aggregated layer values
        cache = []
        output_mlcw_byLayer = pd.DataFrame(data=None, index=None)

        # Loop through each classified layer to sum up individual multi-depth sensor readings
        for idx, select_layer in enumerate(all_layers[:]):

            # Final-Layer Safeguard: Assign any remaining unmapped depths to the bottom layer
            # to ensure strict continuity and zero tracking omissions.
            if idx == (len(all_layers) - 1):
                depth_to_extract = mlcw_reconst_df.columns.difference(cache)
                depth_byLayer_str = classify_df.query(
                    "depth in @depth_to_extract"
                )["depth"].tolist()

            # Standard Operation: Isolate sensor depths explicitly designated for the current active layer
            else:
                depth_byLayer_str = classify_df.query("layer==@select_layer")[
                    "depth"
                ].tolist()

            # Slice displacement columns and sum them horizontally to establish a cumulative layer-wide metric
            mlcw_byDepth = mlcw_reconst_df.loc[:, depth_byLayer_str]
            mlcw_byDepth_sum = mlcw_byDepth.sum(axis=1)
            mlcw_byDepth_sum.name = select_layer

            # Horizontally merge the current layer's time-series into the active processing DataFrame
            output_mlcw_byLayer = pd.concat(
                [output_mlcw_byLayer, mlcw_byDepth_sum], axis=1
            )

            # Log these processed sensor depths into the cache to mask them in the final iteration loop
            cache.extend(depth_byLayer_str)

        # ------------------------------------------------------------------------------
        # PURPOSE: FORMAT DATAFRAMES AND EXPORT PRODUCTION DATASETS TO DISK
        # ------------------------------------------------------------------------------

        # Flatten the DatetimeIndex back into a normal column and rename it formally to "datetime"
        output_mlcw_byLayer = output_mlcw_byLayer.reset_index()
        output_mlcw_byLayer = output_mlcw_byLayer.rename(
            {"index": "datetime"}, axis=1
        )

        # Save the processed layer-by-layer cumulative compaction time-series dataset to a CSV file
        output_mlcw_byLayer.to_csv(
            os.path.join(
                # output_savefld, f"{select_station}_reconst_grouped.csv"
                output_savefld,
                f"{select_station}_modeled_grouped.csv",
            ),
            index=False,
        )

        # Export the standalone hydrogeological stratigraphy/depth mapping table for future model reference
        classify_df.to_csv(
            os.path.join(
                output_savefld, f"{select_station}_classify_table.csv"
            ),
            index=False,
        )
    except Exception as e:
        print(select_station, e)
        pass

  0%|          | 0/39 [00:00<?, ?it/s]

JINHU_XIN "None of [Index(['1.41', '7.153', '19.145', '27.15', '35.126', '42.634'], dtype='object')] are in the [columns]"
LUNFENG_XIN "['14.823'] not in index"


# .

In [ ]:
pd.read_feather(
    r"D:\1000_SCRIPTS\004_Project003\20260427_InSAR_MLCW_v2\data\gwl\well_timeseries\TUKU_gwl_timeseries.feather"
)